# Tutorial 01: Basic Discovery & Federated Ingestion

Welcome to **Tutorial 01** of the `als-finder` suite! In this module, you will learn how to:
1. Verify your environment and extract the bundled Lake Tahoe Basin sample Region of Interest (ROI).
2. Run a federated geospatial search across multiple federal and academic LiDAR repositories (**USGS 3DEP**, **NOAA Coastal**, and **OpenTopography**).
3. Inspect the resulting catalog outputs: `manifest.json`, `catalog.gpkg`, and `catalog.csv`.
4. Preview a dry-run download matrix (`fetch_array.csv`) prior to physical data retrieval.

---

## 1. Environment Verification & CLI Help

Let's ensure `als-finder` is properly installed and accessible:

In [ ]:
%%bash
als-finder --version
als-finder --help

---

## 2. Extracting the Example Region of Interest (ROI)

`als-finder` comes with a bundled sample GeoPackage boundary: the **Lake Tahoe Basin Management Unit (LTBMU)**.
We can extract it to our working directory using the `get-example-roi` command:

In [ ]:
!als-finder get-example-roi

### Inspecting with GeoPandas:

In [ ]:
import geopandas as gpd

roi_gdf = gpd.read_file("ltbmu_boundary.gpkg")
print(f"CRS: {roi_gdf.crs}")
print(f"Total Bounds (WGS84): {roi_gdf.total_bounds}")
roi_gdf.head()

---

## 3. Executing a Federated Search

Now let's search across **USGS 3DEP** and **NOAA Coastal** (and OpenTopography if you have configured an API key) for high-density point clouds (`--density QL1` = $\ge 8.0 \text{ pts/m}^2$).

We specify `--workspace ./demo_workspace/` to isolate all generated metadata into its own directory:

In [ ]:
%%bash
als-finder search \
    --roi ./ltbmu_boundary.gpkg \
    --density QL1 \
    --date "2018-01-01/2024-12-31" \
    --workspace ./demo_workspace/ \
    --provider USGS_EPT \
    --provider NOAA_STAC

---

## 4. Exploring the Generated Catalog Outputs

The search command generated three primary catalog files inside `./demo_workspace/catalog/`:
1. `manifest.json`: Full master metadata document.
2. `catalog.gpkg`: Vector layer with precise polygon bounds of each dataset.
3. `catalog.csv`: Summary table for spreadsheet analysis.

In [ ]:
import json
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

# 1. Read CSV Summary
catalog_df = pd.read_csv("demo_workspace/catalog/catalog.csv")
display(catalog_df[["Provider", "Name", "Date", "PointDensity"]].head())

# 2. Visualize Coverage
roi_gdf = gpd.read_file("ltbmu_boundary.gpkg")
catalog_gdf = gpd.read_file("demo_workspace/catalog/catalog.gpkg")

fig, ax = plt.subplots(figsize=(10, 10))
roi_gdf.plot(ax=ax, color="none", edgecolor="black", linewidth=2, label="ROI (Tahoe Basin)")
catalog_gdf.plot(ax=ax, column="Name", alpha=0.5, legend=True, cmap="Set2")
ax.set_title("Discovered LiDAR Acquisitions Over Lake Tahoe Basin", fontsize=14)
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

---

## 5. Dry-Run Fetch Array Preview

Before downloading gigabytes of point cloud data, `als-finder download` generates a **Dry-Run Fetch Matrix** (`fetch_array.csv`).
This previews the number of physical tiles, estimated download sizes, and target file paths without writing data:

In [ ]:
%%bash
als-finder download \
    --workspace ./demo_workspace/ \
    --roi ./ltbmu_boundary.gpkg

---

### 👉 Next Step
Open [02_normalization_and_stac.md](file:///mnt/c/Users/gears/git/als-finder/tutorials/02_normalization_and_stac.md) (or `02_normalization_and_stac.ipynb`) to learn how to apply SMRF ground classification, compute Height Above Ground (HAG), and generate OGC STAC catalogs!